# 66 - Early Fusion across All Benchmarks

Melengkapi Skema 1 dengan **Early Fusion** (HAE-Net style) untuk semua benchmark dataset, sebelumnya hanya Primer yang di-train (nb 64).

**Strategi**:
- B1 baseline saja (apple-to-apple dengan Skema 1 benchmark convention)
- Kedua backbone: scratch (`EmotionEarlyFusion`) + TL (`EmotionEarlyFusionTransfer`)
- 4 dataset × 2 class × 2 backbone = **16 configs**
- Input: RGB image + landmark heatmap → 4-channel (224x224x4)

**Prerequisites**:
1. Heatmap harus sudah di-generate untuk semua benchmark:
   ```
   python scripts/generate_landmark_heatmaps.py --only 'CK+'
   python scripts/generate_landmark_heatmaps.py --only 'JAFFE'
   python scripts/generate_landmark_heatmaps.py --only 'RAF-DB'
   python scripts/generate_landmark_heatmaps.py --only 'KDEF'
   ```
2. Otomatis skip combo yang heatmap-nya tidak tersedia (dengan warning).

**Output**: update `models/benchmark/{ds}/{ds}_{num}c_results.json` dengan key `EarlyFusion_B1` dan `EarlyFusion_TL_B1`.

**Datasets**: CK+ 7c/4c, JAFFE 7c/4c, RAF-DB 7c/4c, KDEF 7c/4c — **8 variants, 16 runs total**.

**Naming convention**:
- `ckplus`/`jaffe`: `models/benchmark/{ds}/{ds}_{num}c/EarlyFusion_{B1|TL_B1}/`
- `rafdb`/`kdef`: `models/benchmark/{ds}/{num}c/EarlyFusion_{B1|TL_B1}/`

Untuk CK+ 4-class, dataset `ckplus_4class_contempt` digunakan (konsisten dengan nb 65).

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import EmotionEarlyFusion, EmotionEarlyFusionTransfer
from training.utils import train_model, full_evaluation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
LR_SCRATCH = 0.0001
LR_TL = 0.00005

BENCHMARK_DIR = PROJECT_ROOT / 'data' / 'benchmark'
MODELS_DIR = PROJECT_ROOT / 'models' / 'benchmark'

print('Setup complete.')

Device: cuda
GPU: Tesla T4
Setup complete.


In [2]:
# ── Helpers ──

def _subject_split(subjects, seed=42, train_ratio=0.8, val_ratio=0.1):
    rng = np.random.RandomState(seed)
    uniq = np.array(sorted(set(subjects.tolist())))
    rng.shuffle(uniq)
    n = len(uniq)
    n_tr = int(n * train_ratio)
    n_v = int(n * val_ratio)
    return set(uniq[:n_tr].tolist()), set(uniq[n_tr:n_tr+n_v].tolist()), set(uniq[n_tr+n_v:].tolist())


def _dataset_dir(dataset_name, num_classes):
    """Return the on-disk folder. CK+ 4c uses contempt variant (consistent with nb 65)."""
    if dataset_name == 'ckplus' and num_classes == 4:
        return BENCHMARK_DIR / 'ckplus_4class_contempt'
    return BENCHMARK_DIR / f'{dataset_name}_{num_classes}class'


def _stack_4ch(img, heat):
    """Concat (N,224,224,3) image + (N,224,224) heatmap -> (N,224,224,4) float32."""
    if heat.ndim == 3:
        heat = heat[..., None]
    return np.concatenate([img, heat], axis=-1).astype(np.float32, copy=False)


def load_benchmark(dataset_name, num_classes):
    """Load train/val/test 4-channel (image + heatmap) + labels.
    Returns (tr_x4, tr_y, v_x4, v_y, te_x4, te_y) or None if heatmap missing.
    """
    d = _dataset_dir(dataset_name, num_classes)

    if dataset_name in ('rafdb',):
        need = [d / 'X_train_heatmaps.npy', d / 'X_test_heatmaps.npy']
        if not all(p.exists() for p in need):
            print(f'  [SKIP] {dataset_name}_{num_classes}c: missing heatmap files')
            return None
        tr_img = np.load(d / 'X_train_images.npy')
        tr_heat = np.load(d / 'X_train_heatmaps.npy')
        tr_y = np.load(d / 'y_train.npy')
        te_img = np.load(d / 'X_test_images.npy')
        te_heat = np.load(d / 'X_test_heatmaps.npy')
        te_y = np.load(d / 'y_test.npy')
        tr_x4 = _stack_4ch(tr_img, tr_heat)
        te_x4 = _stack_4ch(te_img, te_heat)
        idx_tr, idx_v = train_test_split(np.arange(len(tr_y)), test_size=0.1,
                                         stratify=tr_y, random_state=42)
        return (tr_x4[idx_tr], tr_y[idx_tr], tr_x4[idx_v], tr_y[idx_v], te_x4, te_y)

    if dataset_name == 'kdef':
        need = [d / 'X_train_heatmaps.npy', d / 'X_val_heatmaps.npy', d / 'X_test_heatmaps.npy']
        if not all(p.exists() for p in need):
            print(f'  [SKIP] {dataset_name}_{num_classes}c: missing heatmap files')
            return None
        tr_x4 = _stack_4ch(np.load(d / 'X_train_images.npy'), np.load(d / 'X_train_heatmaps.npy'))
        v_x4 = _stack_4ch(np.load(d / 'X_val_images.npy'), np.load(d / 'X_val_heatmaps.npy'))
        te_x4 = _stack_4ch(np.load(d / 'X_test_images.npy'), np.load(d / 'X_test_heatmaps.npy'))
        return (tr_x4, np.load(d / 'y_train.npy'),
                v_x4, np.load(d / 'y_val.npy'),
                te_x4, np.load(d / 'y_test.npy'))

    if dataset_name in ('ckplus', 'jaffe'):
        heat_path = d / 'X_heatmaps.npy'
        if not heat_path.exists():
            print(f'  [SKIP] {dataset_name}_{num_classes}c: missing {heat_path.name}')
            return None
        img = np.load(d / 'X_images.npy')
        heat = np.load(heat_path)
        y = np.load(d / 'y_labels.npy')
        subjects = np.load(d / 'subjects.npy', allow_pickle=True)
        x4 = _stack_4ch(img, heat)
        tr_subs, v_subs, te_subs = _subject_split(subjects)
        tr_idx = np.where(np.isin(subjects, list(tr_subs)))[0]
        v_idx = np.where(np.isin(subjects, list(v_subs)))[0]
        te_idx = np.where(np.isin(subjects, list(te_subs)))[0]
        return (x4[tr_idx], y[tr_idx], x4[v_idx], y[v_idx], x4[te_idx], y[te_idx])

    raise ValueError(f'Unknown dataset {dataset_name}')


def checkpoint_dir(dataset_name, num_classes, model_key):
    """Match convention: ckplus/jaffe use nested folder, rafdb/kdef flat."""
    if dataset_name in ('ckplus', 'jaffe'):
        return MODELS_DIR / dataset_name / f'{dataset_name}_{num_classes}c' / model_key
    return MODELS_DIR / dataset_name / f'{num_classes}c' / model_key


def results_file(dataset_name, num_classes):
    return MODELS_DIR / dataset_name / f'{dataset_name}_{num_classes}c_results.json'


def make_loader(x4, y, batch_size=BATCH_SIZE, shuffle=True):
    # (N, 224, 224, 4) -> (N, 4, 224, 224)
    t = torch.from_numpy(x4).permute(0, 3, 1, 2).contiguous()
    ys = torch.from_numpy(y).long()
    ds = TensorDataset(t, ys)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


def metrics_triple(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
    }


def train_early_fusion(dataset_name, num_classes, model_key, model_class, lr,
                       tr_x4, tr_y, v_x4, v_y, te_x4, te_y):
    save_dir = checkpoint_dir(dataset_name, num_classes, model_key)
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / 'model.pth'

    model = model_class(num_classes=num_classes).to(device)

    if save_path.exists():
        print(f'    [SKIP train] loading existing {save_path.name}')
        model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    else:
        print(f'    [TRAIN] {model_key} ...')
        tr_loader = make_loader(tr_x4, tr_y, BATCH_SIZE, True)
        val_loader = make_loader(v_x4, v_y, BATCH_SIZE, False)
        crit = nn.CrossEntropyLoss()
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5,
                                                         patience=8, min_lr=1e-7)
        train_model(model, tr_loader, val_loader, crit, opt, sch,
                    device, 'cnn', EPOCHS, PATIENCE, str(save_path))
        model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))

    # Eval on test
    test_loader = make_loader(te_x4, te_y, BATCH_SIZE, False)
    model.eval()
    y_pred = []
    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(device)
            y_pred.append(model(xb).argmax(dim=1).cpu().numpy())
    y_pred = np.concatenate(y_pred)
    return metrics_triple(te_y, y_pred)


def early_fusion_benchmark(dataset_name, num_classes):
    print(f"\n{'='*70}")
    print(f'  Early Fusion — {dataset_name.upper()} {num_classes}c')
    print(f"{'='*70}")

    loaded = load_benchmark(dataset_name, num_classes)
    if loaded is None:
        return None
    tr_x4, tr_y, v_x4, v_y, te_x4, te_y = loaded
    print(f'  Train={len(tr_y)}  Val={len(v_y)}  Test={len(te_y)}  shape={tr_x4.shape[1:]}')

    # Update JSON with both B1 scratch and TL
    rf = results_file(dataset_name, num_classes)
    existing = {}
    if rf.exists():
        with open(rf) as f:
            existing = json.load(f)

    combos = [
        ('EarlyFusion_B1',    EmotionEarlyFusion,         LR_SCRATCH),
        ('EarlyFusion_TL_B1', EmotionEarlyFusionTransfer, LR_TL),
    ]
    out = {}
    for key, model_cls, lr in combos:
        r = train_early_fusion(dataset_name, num_classes, key, model_cls, lr,
                               tr_x4, tr_y, v_x4, v_y, te_x4, te_y)
        out[key] = r
        existing[key] = r
        print(f"    {key:<20} Macro={r['macro_f1']:.4f}  Micro={r['micro_f1']:.4f}  Weighted={r['weighted_f1']:.4f}")

    rf.parent.mkdir(parents=True, exist_ok=True)
    with open(rf, 'w') as f:
        json.dump(existing, f, indent=2)
    print(f'  Updated: {rf}')
    return out


print('Helpers ready.')

Helpers ready.


## Run Early Fusion across 4 Benchmark Datasets × 2 Class Configs

Primer conf60 **tidak di-run** di sini (sudah dilakukan di nb 64, dengan skenario lengkap B1/B2/B3 scratch+TL).

In [3]:
all_results = {}

# CK+ (subject-wise 80/10/10 split from subjects.npy, consistent with nb 65)
all_results['ckplus_7c'] = early_fusion_benchmark('ckplus', 7)
all_results['ckplus_4c'] = early_fusion_benchmark('ckplus', 4)  # uses ckplus_4class_contempt


  Early Fusion — CKPLUS 7c


  Train=508  Val=69  Test=59  shape=(224, 224, 4)


    [TRAIN] EarlyFusion_B1 ...
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      1.9930     0.1949     1.8179    0.5217   0.0980   0.000100  (3.8s)


     2      1.8529     0.2736     1.6612    0.5217   0.0980   0.000100  (3.3s)


...

    40      0.0283     0.9962     0.4516    0.8750   0.8428   0.000025  (1.8s)


    41      0.0532     0.9885     0.4881    0.8333   0.7657   0.000025  (1.8s)


    42      0.0436     0.9962     0.4857    0.8472   0.7552   0.000025  (1.8s)


    43      0.0325     0.9962     0.4045    0.8611   0.7694   0.000025  (1.8s)

Early stopping at epoch 43. Best epoch: 28 (val_f1=0.8457)

Best: epoch 28, val_acc=0.9028, val_f1=0.8457
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/ckplus/ckplus_4c/EarlyFusion_TL_B1/model.pth


    EarlyFusion_TL_B1    Macro=0.7952  Micro=0.8710  Weighted=0.8717
  Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/ckplus/ckplus_4c_results.json


In [4]:
# JAFFE (subject-wise split)
all_results['jaffe_7c'] = early_fusion_benchmark('jaffe', 7)
all_results['jaffe_4c'] = early_fusion_benchmark('jaffe', 4)


  Early Fusion — JAFFE 7c


  Train=173  Val=20  Test=20  shape=(224, 224, 4)


    [TRAIN] EarlyFusion_B1 ...
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      2.0655     0.1329     1.9489    0.1500   0.0373   0.000100  (1.1s)


     2      1.9784     0.1792     1.9671    0.1500   0.0373   0.000100  (1.1s)


...

    28      0.0748     1.0000     0.9975    0.6500   0.5614   0.000025  (0.6s)


    29      0.0712     1.0000     1.0040    0.6500   0.5614   0.000025  (0.6s)


    30      0.0697     1.0000     0.9984    0.6500   0.5614   0.000025  (0.6s)


    31      0.0724     1.0000     1.0186    0.6000   0.5342   0.000025  (0.6s)


    32      0.0712     1.0000     1.0010    0.6500   0.5614   0.000025  (0.6s)

Early stopping at epoch 32. Best epoch: 17 (val_f1=0.5631)

Best: epoch 17, val_acc=0.6500, val_f1=0.5631
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/jaffe/jaffe_4c/EarlyFusion_TL_B1/model.pth
    EarlyFusion_TL_B1    Macro=0.3519  Micro=0.6000  Weighted=0.5074
  Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/jaffe/jaffe_4c_results.json


In [5]:
# RAF-DB (official train/test split + 10% val from train)
all_results['rafdb_7c'] = early_fusion_benchmark('rafdb', 7)
all_results['rafdb_4c'] = early_fusion_benchmark('rafdb', 4)


  Early Fusion — RAFDB 7c


  Train=10408  Val=1157  Test=2884  shape=(224, 224, 4)


    [TRAIN] EarlyFusion_B1 ...


 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      1.4926     0.4597     1.1372    0.6024   0.3682   0.000100  (68.3s)


...

    36      0.0045     0.9992     0.6622    0.8401   0.8148   0.000013  (35.0s)


    37      0.0034     0.9995     0.6569    0.8332   0.8089   0.000013  (35.0s)


    38      0.0023     0.9997     0.6646    0.8341   0.8100   0.000013  (35.0s)


    39      0.0020     0.9998     0.6480    0.8332   0.8079   0.000013  (35.0s)

Early stopping at epoch 39. Best epoch: 24 (val_f1=0.8188)

Best: epoch 24, val_acc=0.8410, val_f1=0.8188
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/rafdb/4c/EarlyFusion_TL_B1/model.pth


    EarlyFusion_TL_B1    Macro=0.7992  Micro=0.8228  Weighted=0.8232
  Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/rafdb/rafdb_4c_results.json


In [6]:
# KDEF (explicit train/val/test split)
all_results['kdef_7c'] = early_fusion_benchmark('kdef', 7)
all_results['kdef_4c'] = early_fusion_benchmark('kdef', 4)


  Early Fusion — KDEF 7c


  Train=2631  Val=340  Test=337  shape=(224, 224, 4)


    [TRAIN] EarlyFusion_B1 ...


 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      2.0232     0.1737     1.8425    0.2647   0.1880   0.000100  (16.4s)


...

    17      0.0188     0.9985     0.4661    0.8588   0.8314   0.000025  (9.2s)


    18      0.0151     1.0000     0.4599    0.8471   0.8167   0.000025  (9.1s)


    19      0.0160     0.9992     0.4762    0.8471   0.8087   0.000025  (9.4s)


    20      0.0254     0.9970     0.4963    0.8471   0.8227   0.000025  (9.1s)

Early stopping at epoch 20. Best epoch: 5 (val_f1=0.8315)

Best: epoch 5, val_acc=0.8559, val_f1=0.8315
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/kdef/4c/EarlyFusion_TL_B1/model.pth


    EarlyFusion_TL_B1    Macro=0.8157  Micro=0.8546  Weighted=0.8544
  Updated: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/kdef/kdef_4c_results.json


## Ringkasan Early Fusion (8 Benchmark Variants × 2 Backbones)

In [7]:
print(f"\n{'='*82}")
print(f'  Early Fusion — Summary across 8 benchmark variants')
print(f"{'='*82}")
print(f"  {'Dataset':<14} {'Config':<22} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Acc':>10}")
print(f"  {'-'*80}")
for ds_key, res in all_results.items():
    if res is None:
        print(f'  {ds_key:<14} (skipped — no heatmap)')
        continue
    for cfg_key in ['EarlyFusion_B1', 'EarlyFusion_TL_B1']:
        r = res[cfg_key]
        print(f"  {ds_key:<14} {cfg_key:<22} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} {r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f}")


  Early Fusion — Summary across 8 benchmark variants
  Dataset        Config                      Macro      Micro   Weighted        Acc
  --------------------------------------------------------------------------------
  ckplus_7c      EarlyFusion_B1             0.4458     0.6949     0.6646     0.6949
  ckplus_7c      EarlyFusion_TL_B1          0.7624     0.8475     0.8471     0.8475
  ckplus_4c      EarlyFusion_B1             0.5075     0.6935     0.6666     0.6935
  ckplus_4c      EarlyFusion_TL_B1          0.7952     0.8710     0.8717     0.8710
  jaffe_7c       EarlyFusion_B1             0.2857     0.3500     0.2667     0.3500
  jaffe_7c       EarlyFusion_TL_B1          0.0408     0.1500     0.0429     0.1500
  jaffe_4c       EarlyFusion_B1             0.1774     0.5500     0.3903     0.5500
  jaffe_4c       EarlyFusion_TL_B1          0.3519     0.6000     0.5074     0.6000
  rafdb_7c       EarlyFusion_B1             0.7098     0.8079     0.8044     0.8079
  rafdb_7c       EarlyF